In [3]:

#importing needed modules 
import os
import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

# Load environment variables from a .env file (ensure .env is in your .gitignore)
load_dotenv()

# Set core IDs; secrets are retrieved from the environment for safety
os.environ["AZURE_TENANT_ID"] = "4cf91b0a-30d9-4e28-88dc-fb108d00fd04"
os.environ["AZURE_CLIENT_ID"] = "1a04d896-f727-4a7d-8ee5-11c3ce0f78ef"

# Fetch token using DefaultAzureCredential
try:
    cred = DefaultAzureCredential()
    at = cred.get_token("https://management.azure.com/.default")
    access_token = at.token
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    print("✅ Token acquired successfully from Azure")
except Exception as e:
    print(f"❌ Failed to acquire token: {e}")

✅ Token acquired successfully from Azure


In [11]:
# defining parameters 

subscriptionId = "11d701d8-f4aa-4e6a-810b-c4dc3ad3940f"
vmName = "mycodeVM"
nic_name = "myNic"
vnet_name = "vnet-southindia"
subnet_name = "snet-southindia-1"
resourceGroupName = "your-target-rg" 
location = "southindia"

In [13]:

## here we are trying to fetch all the resource groups available in our subscription and based on that we will choose region and resource gorup to deploy our VM 
url = f"https://management.azure.com/subscriptions/{subscriptionId}/resourcegroups?api-version=2021-04-01"
# here we are constructing our header which is with help of access token generated above ,header basically does authentication and authorization
#make sure your service principal has needed rights on azure to perform operations 
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

# here we making a get request with needed header and URL to fetch all resource groups available 

response = requests.get(url, headers=headers)

print(response.json()) # This returns your dictionary! which contains needed information like location,tags,name etc which can be used further 

{'value': [{'id': '/subscriptions/11d701d8-f4aa-4e6a-810b-c4dc3ad3940f/resourceGroups/rg-prod-applications', 'name': 'rg-prod-applications', 'type': 'Microsoft.Resources/resourceGroups', 'location': 'eastus', 'tags': {}, 'properties': {'provisioningState': 'Succeeded'}}, {'id': '/subscriptions/11d701d8-f4aa-4e6a-810b-c4dc3ad3940f/resourceGroups/rg-dev-applications', 'name': 'rg-dev-applications', 'type': 'Microsoft.Resources/resourceGroups', 'location': 'eastus', 'tags': {}, 'properties': {'provisioningState': 'Succeeded'}}, {'id': '/subscriptions/11d701d8-f4aa-4e6a-810b-c4dc3ad3940f/resourceGroups/rg-prod-networking', 'name': 'rg-prod-networking', 'type': 'Microsoft.Resources/resourceGroups', 'location': 'westus', 'tags': {}, 'properties': {'provisioningState': 'Succeeded'}}, {'id': '/subscriptions/11d701d8-f4aa-4e6a-810b-c4dc3ad3940f/resourceGroups/rg-dev1-applications', 'name': 'rg-dev1-applications', 'type': 'Microsoft.Resources/resourceGroups', 'location': 'eastus', 'tags': {}, 'p

In [23]:
#get name and location of each resource group 
rg=response.json()
for i in range(len(rg['value'])):
        print(rg['value'][i]['location'])
        print(rg['value'][i]['name'])

eastus
rg-prod-applications
eastus
rg-dev-applications
westus
rg-prod-networking
eastus
rg-dev1-applications
southindia
compute
southindia
NetworkWatcherRG


In [27]:
choice=input("which regioon to deploy resources")
             

which regioon to deploy resources southindia


In [31]:
rgs=rg
def rgfetch(x):
    for i in range(len(rg['value'])):
        if rg['value'][i]['location']==x:
            return [rg['value'][i]['name'],rg['value'][i]['location']]
rgfetch(choice)[0]

'compute'

In [37]:
subscriptionId = "11d701d8-f4aa-4e6a-810b-c4dc3ad3940f"
vmName = "mycodeVM"
nic_name = "myNic"
vnet_name = "vnet-southindia"
subnet_name = "snet-southindia-1"
resourceGroupName = rgfetch(choice)[0] 
location = rgfetch(choice)[1]

In [39]:
#Network Interface (NIC) Creation

nic_url = f"https://management.azure.com/subscriptions/{subscriptionId}/resourceGroups/{resourceGroupName}/providers/Microsoft.Network/networkInterfaces/{nic_name}?api-version=2023-09-01"

nic_data = {
    "location": location,
    "properties": {
        "ipConfigurations": [{
            "name": "ipconfig1",
            "properties": {
                "subnet": {
                    "id": f"/subscriptions/{subscriptionId}/resourceGroups/{resourceGroupName}/providers/Microsoft.Network/virtualNetworks/{vnet_name}/subnets/{subnet_name}"
                },
                "privateIPAllocationMethod": "Dynamic"
            }
        }]
    }
}

response_nic = requests.put(nic_url, headers=headers, json=nic_data)

if response_nic.status_code in [200, 201]:
    nic_full_id = response_nic.json()['id']
    print(f"✅ NIC Created! ID: {nic_full_id}")
else:
    print(f"❌ NIC Creation Failed: {response_nic.text}")

✅ NIC Created! ID: /subscriptions/11d701d8-f4aa-4e6a-810b-c4dc3ad3940f/resourceGroups/compute/providers/Microsoft.Network/networkInterfaces/myNic


In [41]:
#now lets create Virtual machine using this nic id 

# Fixed JSON structure with correct nesting and lowercase keys
vm_data = {
    "location": location,
    "properties": {
        "hardwareProfile": {
            "vmSize": "Standard_D2ads_v5"
        },
        "storageProfile": {
            "imageReference": {
                "publisher": "Canonical",
                "offer": "UbuntuServer",
                "sku": "18.04-LTS",
                "version": "latest"
            },
            "osDisk": {
                "name": f"{vmName}_osDisk",
                "caching": "ReadWrite",
                "createOption": "FromImage",
                "managedDisk": {
                    "storageAccountType": "Standard_LRS"
                }
            }
        },
        "osProfile": {
            "computerName": vmName,
            "adminUsername": "azureuser",
            "adminPassword": "ReplaceWithSecurePassword123!" # Never commit real passwords
        },
        "networkProfile": {
            "networkInterfaces": [
                {
                    "id": nic_full_id 
                }
            ]
        }
    }
}

vm_url = f"https://management.azure.com/subscriptions/{subscriptionId}/resourceGroups/{resourceGroupName}/providers/Microsoft.Compute/virtualMachines/{vmName}?api-version=2024-07-01"


In [43]:

response_vm = requests.put(vm_url, headers=headers, json=vm_data)

if response_vm.status_code in [200, 201, 202]:
    print(f"🚀 VM creation initiated: {vmName}")
else:
    print(f"❌ VM Creation Failed: {response_vm.text}")

🚀 VM creation initiated: mycodeVM
